In [ ]:
import pandas as pd

# Kaggle 會自動把資料放在這個路徑
TRAIN_PATH = '/kaggle/input/titanic/train.csv'
TEST_PATH = '/kaggle/input/titanic/test.csv'

train_data = pd.read_csv(TRAIN_PATH)
test_data = pd.read_csv(TEST_PATH)

print("✅ Kaggle data successfully loaded!")
print("--- 訓練資料 (前 5 行) ---")
print(train_data.head())

In [ ]:
print("--- 1. 資料總體報告 (info) ---")
# .info() 會印出非常詳細的摘要
train_data.info()

In [ ]:
print("\n--- 2. 各欄位缺失值統計 (isnull().sum()) ---")
print(train_data.isnull().sum())

In [ ]:
print("\n--- 3. 數值型資料統計摘要 (describe) ---")
# .describe() 只會分析數字欄位
train_data.describe()

In [ ]:
print("\n--- 4. 類別型資料內容統計 (value_counts) ---")

print("\n(a) 性別 (Sex) 分佈：")
print(train_data['Sex'].value_counts())

print("\n(b) 登船港口 (Embarked) 分佈：")
print(train_data['Embarked'].value_counts())

print("\n(c) 乘客艙等 (Pclass) 分佈：")
print(train_data['Pclass'].value_counts())

In [ ]:
import pandas as pd
import numpy as np
import re 

# --- A. 重新載入所有資料 (確保是乾淨的原始狀態) ---
train_data = pd.read_csv('/kaggle/input/titanic/train.csv')
test_data = pd.read_csv('/kaggle/input/titanic/test.csv')

# --- B. 從「訓練資料 (train_data)」中學習「所有清理規則」 ---
print("--- 正在從 train_data 學習「清理規則」... ---")

# 規則 1: 學習 'Title' 和 'Age' 的填充規則
train_data['Title'] = train_data['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
common_titles = ['Mr', 'Miss', 'Mrs', 'Master']
train_data['Title'] = train_data['Title'].fillna('Other').apply(lambda x: x if x in common_titles else 'Other')
AGE_FILL_MAP = train_data.groupby('Title')['Age'].median().to_dict()

# 規則 2: 學習 'Deck' (甲板) 的填充規則
train_data['Deck'] = train_data['Cabin'].apply(lambda s: s[0] if pd.notnull(s) else 'U')
DECK_MAP = train_data.groupby('Ticket')['Deck'].apply(
    lambda x: x[x != 'U'].mode()[0] if not x[x != 'U'].empty else 'U'
).to_dict()

# 規則 3: 學習 'Embarked' (港口) 和 'Fare' (票價) 的填充規則
EMBARKED_FILL = train_data['Embarked'].mode()[0]
FARE_FILL = train_data['Fare'].median()

# 規則 4: 學習「文字 -> 數字」的轉換規則
SEX_MAP = {'male': 0, 'female': 1}
EMBARKED_MAP = {'S': 0, 'C': 1, 'Q': 2}
TITLE_MAP = {'Mr': 0, 'Miss': 1, 'Mrs': 2, 'Master': 3, 'Other': 4}
DECK_MAP_FINAL = {d:i for i, d in enumerate(sorted(train_data['Deck'].unique()))}
DECK_FILL_UNKNOWN = DECK_MAP_FINAL.get('U', 8) 

print("   - 學習到所有「清理規則」。")


# --- C. 【已升級 v2 版】定義一個「萬用清理函式」 ---
#     (新增了 FamilySize 和 IsAlone)

def preprocess_data(df, age_map, deck_ticket_map, embarked_fill, fare_fill, maps):
    df_clean = df.copy() # 複製一份，才不會改到原始資料
    
    # 1. 處理 'Age'
    df_clean['Title'] = df_clean['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    df_clean['Title'] = df_clean['Title'].fillna('Other').apply(lambda x: x if x in common_titles else 'Other')
    df_clean['Age'] = df_clean['Age'].fillna(df_clean['Title'].map(age_map))
    df_clean['Age'] = df_clean['Age'].fillna(df['Age'].median()) 
    
    # 2. 處理 'Deck'
    df_clean['Deck'] = df_clean['Cabin'].apply(lambda s: s[0] if pd.notnull(s) else 'U')
    def fill_deck(row):
        if row['Deck'] == 'U':
            return deck_ticket_map.get(row['Ticket'], 'U')
        return row['Deck']
    df_clean['Deck'] = df_clean.apply(fill_deck, axis=1)

    # 3. 處理 'Embarked' 和 'Fare'
    df_clean['Embarked'] = df_clean['Embarked'].fillna(embarked_fill)
    df_clean['Fare'] = df_clean['Fare'].fillna(fare_fill)

    # 4. 【新特徵】FamilySize 和 IsAlone
    #    (在轉換文字前，先從 SibSp 和 Parch 計算)
    df_clean['FamilySize'] = df_clean['SibSp'] + df_clean['Parch'] + 1
    df_clean['IsAlone'] = (df_clean['FamilySize'] == 1).astype(int) # 轉換 True/False 為 1/0
    
    # 5. 轉換文字
    df_clean['Sex'] = df_clean['Sex'].map(maps['sex'])
    df_clean['Embarked'] = df_clean['Embarked'].map(maps['embarked'])
    df_clean['Title'] = df_clean['Title'].map(maps['title'])
    df_clean['Deck'] = df_clean['Deck'].map(maps['deck'])
    df_clean['Deck'] = df_clean['Deck'].fillna(maps['deck_unknown'])

    return df_clean

print("\n✅ --- 【v2 版】清理函式 `preprocess_data_v2` 建立完成！ ---")

In [ ]:
# 1. 儲存我們學到的所有「規則」
all_rules = {
    "age_map": AGE_FILL_MAP,
    "deck_ticket_map": DECK_MAP,
    "embarked_fill": EMBARKED_FILL,
    "fare_fill": FARE_FILL,
    "maps": {
        "sex": SEX_MAP,
        "embarked": EMBARKED_MAP,
        "title": TITLE_MAP,
        "deck": DECK_MAP_FINAL,
        "deck_unknown": DECK_FILL_UNKNOWN
    }
}

# 2. 【清理資料】(使用 v2 函式)
print("--- 正在清理 train_data... ---")
train_clean = preprocess_data(train_data, **all_rules) 
print("--- 正在清理 test_data... ---")
test_clean = preprocess_data(test_data, **all_rules)

# 3. 【準備 X 和 y】(加入新特徵 FamilySize 和 IsAlone)
test_clean['Title'] = test_clean['Title'].fillna(0) # 確保 test_clean 也沒有 NaN
test_clean['Deck'] = test_clean['Deck'].fillna(0)

# 我們的「最終版」特徵清單
features= ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'Title', 'Deck', 'FamilySize', 'IsAlone']
X_train = train_clean[features]
y_train = train_clean['Survived']
X_test = test_clean[features]

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# --- 1. 【準備資料】 ---
# (我們使用 v2 函式清理過、但「尚未 OHE」的資料)
X_train_pre_ohe = train_clean[features] 
y_train = train_clean['Survived']
X_test_pre_ohe = test_clean[features]

# --- 2. 【關鍵：OHE (獨熱編碼)】 ---
print("--- 1. 正在執行 One-Hot Encoding... ---")

# 2a. 【合併】為了確保欄位一致，先合併 train 和 test
combined_df = pd.concat([X_train_pre_ohe, X_test_pre_ohe], axis=0)

# 2b. 定義哪些欄位是「類別」(我們要 OHE 的)
categorical_features = ['Pclass', 'Sex', 'Embarked', 'Title', 'Deck', 'IsAlone']
# (FamilySize, Age, Fare, SibSp, Parch 是「數值」，我們保留)

# 2c. 【執行 OHE】
#     pd.get_dummies 會自動轉換這些欄位
#     drop_first=True 是為了防止「多重共線性」(例如你只需要 Pclass_2, Pclass_3，就能推斷 Pclass_1)
combined_df_ohe = pd.get_dummies(combined_df, columns=categorical_features, drop_first=True)

# 2d. 【拆分】將合併的資料，依據原始長度拆分回 train 和 test
X_train_ohe = combined_df_ohe.iloc[:len(X_train_pre_ohe)]
X_test_ohe = combined_df_ohe.iloc[len(X_train_pre_ohe):]

print(f"   - 特徵數量從 {len(features)} 個，擴展到了 {len(X_train_ohe.columns)} 個！")


# --- 3. 【特徵縮放 (仍然必要)】 ---
print("--- 2. 正在對「OHE 後」的資料進行特徵縮放... ---")
# 邏輯斯迴歸「依然」需要縮放
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_ohe)
X_test_scaled = scaler.transform(X_test_ohe)


# --- 4. 【訓練「冠軍模型」(邏輯斯迴歸)】 ---
print("--- 3. 正在用「最終版」資料訓練邏輯斯迴歸... ---")
model_lr_final = LogisticRegression(C=1.0, random_state=42, max_iter=1000)
model_lr_final.fit(X_train_scaled, y_train)

# 4a. 評估「訓練集」分數 (看看它是否健康)
train_preds_lr_final = model_lr_final.predict(X_train_scaled)
train_accuracy_lr_final = accuracy_score(y_train, train_preds_lr_final)
print(f"   - (參考) 最終模型在「訓練集」上的準確率: {train_accuracy_lr_final * 100:.2f}%")


# --- 5. 【產生提交檔案】 ---
print("--- 4. 正在產生「最終版」提交檔案... ---")
test_predictions_lr_final = model_lr_final.predict(X_test_scaled)

submission_lr_final = pd.DataFrame({
    "PassengerId": test_data["PassengerId"],
    "Survived": test_predictions_lr_final
})

submission_lr_final.to_csv('submission_logistic_final_ohe.csv', index=False)


print("\n🎉 --- 全新提交檔案 submission_logistic_final_ohe.csv 建立完成！ --- 🎉")


In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# --- 1. 重新載入所有「原始」資料 ---
df_train_raw = pd.read_csv('/kaggle/input/titanic/train.csv')
df_test_raw = pd.read_csv('/kaggle/input/titanic/test.csv')

# 儲存 y_train (我們的目標)
y_train = df_train_raw['Survived']
# 儲存 test PassengerId (用於最後提交)
test_passenger_ids = df_test_raw['PassengerId']

# --- 2. 【關鍵】先合併 (Concat) ---
#     (我們先把 Survived 拿掉，並貼上標籤，方便之後拆分)
df_train_proc = df_train_raw.drop(['Survived'], axis=1)
df_all = pd.concat([df_train_proc, df_test_raw], ignore_index=True)

print(f"--- 1. 已合併 train 和 test 資料，總共 {len(df_all)} 筆 ---")

# --- 3. 【在「合併資料」上執行「特徵工程」】 ---
print("--- 2. 正在執行特徵工程... ---")
# 3a. 創造 'Title'
df_all['Title'] = df_all['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
common_titles = ['Mr', 'Miss', 'Mrs', 'Master']
df_all['Title'] = df_all['Title'].fillna('Other').apply(lambda x: x if x in common_titles else 'Other')

# 3b. 創造 'FamilySize' 和 'IsAlone'
df_all['FamilySize'] = df_all['SibSp'] + df_all['Parch'] + 1
df_all['IsAlone'] = (df_all['FamilySize'] == 1).astype(int)

# 3c. 創造 'Deck'
df_all['Deck'] = df_all['Cabin'].apply(lambda s: s[0] if pd.notnull(s) else 'U')

# 3d. (暫時) 建立 'Ticket' 映射 (我們稍後會刪除 Ticket)
deck_map = df_all.groupby('Ticket')['Deck'].apply(
    lambda x: x[x != 'U'].mode()[0] if not x[x != 'U'].empty else 'U'
).to_dict()
def fill_deck(row):
    if row['Deck'] == 'U':
        return deck_map.get(row['Ticket'], 'U')
    return row['Deck']
df_all['Deck'] = df_all.apply(fill_deck, axis=1)

# --- 4. 【在「合併資料」上執行「填充 (Imputation)」】 ---
print("--- 3. 正在填充所有缺失值... ---")
# 4a. 填充 'Age' (依 'Title' 的中位數)
age_fill_map = df_all.groupby('Title')['Age'].median().to_dict()
df_all['Age'] = df_all['Age'].fillna(df_all['Title'].map(age_fill_map))
df_all['Age'] = df_all['Age'].fillna(df_all['Age'].median()) # 確保 100% 填滿

# 4b. 填充 'Fare' (依「全體」中位數)
df_all['Fare'] = df_all['Fare'].fillna(df_all['Fare'].median())

# 4c. 填充 'Embarked' (依「全體」眾數)
df_all['Embarked'] = df_all['Embarked'].fillna(df_all['Embarked'].mode()[0])

# --- 5. 【關鍵：One-Hot Encoding】 ---
print("--- 4. 正在執行 One-Hot Encoding... ---")
# 5a. 丟棄我們不再需要的「原始」欄位
df_all_proc = df_all.drop(['PassengerId', 'Name', 'Ticket', 'Cabin', 'SibSp', 'Parch'], axis=1)

# 5b. 定義哪些是「類別」(我們要 OHE 的)
categorical_features = ['Pclass', 'Sex', 'Embarked', 'Title', 'Deck', 'IsAlone']
# (FamilySize, Age, Fare 是「數值」，我們會保留)

# 5c. 【執行 OHE】
df_all_ohe = pd.get_dummies(df_all_proc, columns=categorical_features, drop_first=True)

# --- 6. 【拆分並縮放】 ---
print("--- 5. 正在拆分並進行特徵縮放... ---")
# 6a. 拆分回 train 和 test
X_train_final = df_all_ohe.iloc[:len(df_train_proc)]
X_test_final = df_all_ohe.iloc[len(df_train_proc):]

# 6b. 特徵縮放
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_final)
X_test_scaled = scaler.transform(X_test_final)

# --- 7. 【訓練「冠軍模型」(邏輯斯迴歸)】 ---
print("--- 6. 正在用「最終版」資料訓練邏輯斯迴歸... ---")
model_lr_final = LogisticRegression(C=1.0, random_state=42, max_iter=1000)
model_lr_final.fit(X_train_scaled, y_train)

# 7a. 評估「訓練集」分數
train_preds_lr_final = model_lr_final.predict(X_train_scaled)
train_accuracy_lr_final = accuracy_score(y_train, train_preds_lr_final)
print(f"   - (參考) 「終極版」模型在「訓練集」上的準確率: {train_accuracy_lr_final * 100:.2f}%")

# --- 8. 【產生提交檔案】 ---
print("--- 7. 正在產生「終極版」提交檔案... ---")
test_predictions_lr_final = model_lr_final.predict(X_test_scaled)

submission_lr_final = pd.DataFrame({
    "PassengerId": test_passenger_ids,
    "Survived": test_predictions_lr_final
})

submission_lr_final.to_csv('submission_logistic_final_v2_ohe.csv', index=False)

print("\n🎉 --- 全新提交檔案 submission_logistic_final_v2_ohe.csv 建立完成！ --- 🎉")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler # 載入我們的「縮放」工具
from sklearn.metrics import accuracy_score

# ----------------------------------------------------
# 1. 【新增步驟】特徵縮放
#    我們在 X_train 和 X_test 上都執行這個步驟
# ----------------------------------------------------
print("--- 1. 正在進行「特徵縮放」(Scaling)... ---")

# 1a. 建立一個縮放器 (Scaler)
scaler = StandardScaler()

# 1b. 【關鍵】讓縮放器「學習」訓練資料 (X_train) 的平均值和標準差
#     注意：我們「只」在 X_train 上 .fit()，這是黃金準則！
scaler.fit(X_train)

# 1c. 使用學習到的規則，來「轉換」訓練集和測試集
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("   - 特徵縮放完成！")

# ----------------------------------------------------
# 2. 訓練「邏輯斯迴歸」模型
# ----------------------------------------------------
print("\n--- 2. 正在訓練「邏輯斯迴歸」模型... ---")

# 2a. 建立模型 (C=1.0 是預設值，random_state 確保結果可重現)
model_lr = LogisticRegression(C=1.0, random_state=42, max_iter=1000) # 增加 max_iter 確保收斂

# 2b. 【用「縮放後」的資料】訓練模型
model_lr.fit(X_train_scaled, y_train)

print("   - 模型訓練完成！")

# ----------------------------------------------------
# 3. 評估與提交
# ----------------------------------------------------
print("\n--- 3. 正在評估與產生提交檔案... ---")

# 3a. 評估一下「訓練集」上的分數 (你會發現這個分數會比 98% 低，這是好事！)
train_preds_lr = model_lr.predict(X_train_scaled)
train_accuracy_lr = accuracy_score(y_train, train_preds_lr)
print(f"   - (參考) 邏輯斯迴歸在「訓練集」上的準確率: {train_accuracy_lr * 100:.2f}%")

# 3b. 【用「縮放後」的資料】預測「考卷 (X_test)」
test_predictions_lr = model_lr.predict(X_test_scaled)

# 3c. 建立提交檔案
submission_lr = pd.DataFrame({
    "PassengerId": test_data["PassengerId"],
    "Survived": test_predictions_lr
})

# 3d. 儲存檔案
submission_lr.to_csv('submission_logistic.csv', index=False)

print("\n🎉 --- 全新提交檔案 submission_logistic.csv 建立完成！ --- 🎉")
print(submission_lr.head())

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.pipeline import Pipeline             # 載入「管線」工具
from sklearn.preprocessing import StandardScaler    # 載入「縮放」工具
from sklearn.linear_model import LogisticRegression # 載入我們的「冠軍」
from sklearn.model_selection import GridSearchCV  # 載入「網格搜尋」
from sklearn.metrics import accuracy_score

# ===================================================================
# == 1.【重置環境】重新建立我們的「冠軍」資料集 (v2 特徵)
# ===================================================================

# --- A. 重新載入所有資料 (確保是乾淨的原始狀態) ---
train_data = pd.read_csv('/kaggle/input/titanic/train.csv')
test_data = pd.read_csv('/kaggle/input/titanic/test.csv')
test_passenger_ids = test_data['PassengerId'] # 儲存 ID 用於提交

# --- B. 從「訓練資料 (train_data)」中學習「所有清理規則」 ---
print("--- 1. 正在重新學習「v2 清理規則」... ---")
train_data['Title'] = train_data['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
common_titles = ['Mr', 'Miss', 'Mrs', 'Master']
train_data['Title'] = train_data['Title'].fillna('Other').apply(lambda x: x if x in common_titles else 'Other')
AGE_FILL_MAP = train_data.groupby('Title')['Age'].median().to_dict()
train_data['Deck'] = train_data['Cabin'].apply(lambda s: s[0] if pd.notnull(s) else 'U')
DECK_MAP = train_data.groupby('Ticket')['Deck'].apply(
    lambda x: x[x != 'U'].mode()[0] if not x[x != 'U'].empty else 'U'
).to_dict()
EMBARKED_FILL = train_data['Embarked'].mode()[0]
FARE_FILL = train_data['Fare'].median()
SEX_MAP = {'male': 0, 'female': 1}
EMBARKED_MAP = {'S': 0, 'C': 1, 'Q': 2}
TITLE_MAP = {'Mr': 0, 'Miss': 1, 'Mrs': 2, 'Master': 3, 'Other': 4}
DECK_MAP_FINAL = {d:i for i, d in enumerate(sorted(train_data['Deck'].unique()))}
DECK_FILL_UNKNOWN = DECK_MAP_FINAL.get('U', 8) 
all_rules = {
    "age_map": AGE_FILL_MAP, "deck_ticket_map": DECK_MAP, "embarked_fill": EMBARKED_FILL,
    "fare_fill": FARE_FILL, "maps": { "sex": SEX_MAP, "embarked": EMBARKED_MAP, "title": TITLE_MAP,
        "deck": DECK_MAP_FINAL, "deck_unknown": DECK_FILL_UNKNOWN }
}

# --- C. 【v2 版】清理函式 (來自實驗 #3) ---
def preprocess_data_v2(df, age_map, deck_ticket_map, embarked_fill, fare_fill, maps):
    df_clean = df.copy()
    df_clean['Title'] = df_clean['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    df_clean['Title'] = df_clean['Title'].fillna('Other').apply(lambda x: x if x in common_titles else 'Other')
    df_clean['Age'] = df_clean['Age'].fillna(df_clean['Title'].map(age_map))
    df_clean['Age'] = df_clean['Age'].fillna(df['Age'].median()) 
    df_clean['Deck'] = df_clean['Cabin'].apply(lambda s: s[0] if pd.notnull(s) else 'U')
    def fill_deck(row):
        if row['Deck'] == 'U':
            return deck_ticket_map.get(row['Ticket'], 'U')
        return row['Deck']
    df_clean['Deck'] = df_clean.apply(fill_deck, axis=1)
    df_clean['Embarked'] = df_clean['Embarked'].fillna(embarked_fill)
    df_clean['Fare'] = df_clean['Fare'].fillna(fare_fill)
    df_clean['FamilySize'] = df_clean['SibSp'] + df_clean['Parch'] + 1
    df_clean['IsAlone'] = (df_clean['FamilySize'] == 1).astype(int)
    # 【關鍵】我們在這裡「手動」轉換文字 -> 數字 (非 OHE)
    df_clean['Sex'] = df_clean['Sex'].map(maps['sex'])
    df_clean['Embarked'] = df_clean['Embarked'].map(maps['embarked'])
    df_clean['Title'] = df_clean['Title'].map(maps['title'])
    df_clean['Deck'] = df_clean['Deck'].map(maps['deck'])
    df_clean['Deck'] = df_clean['Deck'].fillna(maps['deck_unknown'])
    return df_clean

# --- D. 準備好 X_train, y_train, X_test ---
print("--- 2. 正在準備「v2 特徵」資料集... ---")
train_clean = preprocess_data_v2(train_data, **all_rules) 
test_clean = preprocess_data_v2(test_data, **all_rules)
features_v2 = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'Title', 'Deck', 'FamilySize', 'IsAlone']
X_train = train_clean[features_v2]
y_train = train_clean['Survived']
X_test = test_clean[features_v2]

# (確保 test_clean 中沒有任何意外的 NaN)
X_test = X_test.fillna(X_train.median())

print("✅ --- 「v2 資料集」已準備就緒！ ---")

# ===================================================================
# == 2.【終極調校】Pipeline + GridSearchCV
# ===================================================================

# --- A. 建立「管線」(Pipeline) ---
print("\n--- 3. 正在建立「特徵縮放 + 邏輯斯迴歸」管線... ---")
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),  # 第 1 步：縮放資料
    ('model', LogisticRegression(random_state=42, max_iter=1000, solver='liblinear')) # 第 2 步：訓練模型
])

# --- B. 定義我們要測試的「參數網格」 ---
# (注意：在管線中，參數名稱必須是 "步驟名稱__參數名稱")
param_grid_lr = {
    'model__C': [0.01, 0.1, 0.5, 1.0, 2.0, 5.0]  # 測試 6 種不同的「正規化強度 C」
}

# --- C. 設定「自動調校助理」 ---
print("--- 4. 正在設定 GridSearchCV... ---")
grid_search_lr = GridSearchCV(
    estimator=pipe_lr,
    param_grid=param_grid_lr,
    cv=5,      # 5 折交叉驗證
    n_jobs=-1, # 使用所有 CPU 核心
    verbose=1  # 顯示進度
)

# --- D. 【開始搜尋！】 ---
print("--- 5. 正在自動搜尋「邏輯斯迴歸」的最佳 C 值... ---")
grid_search_lr.fit(X_train, y_train) # 它會自動處理 X_train 的縮放

print("✅ --- 自動搜尋完成！ ---")

# --- E. 【查看「最佳結果」】 ---
print("\n--- 6. 正在報告最佳結果... ---")
print(f"   - 最佳參數組合 (Best params): {grid_search_lr.best_params_}")
print(f"   - 最佳交叉驗證分數 (Best CV score): {grid_search_lr.best_score_ * 100:.2f}%")
best_lr_model = grid_search_lr.best_estimator_

# --- F. 【用「最強模型」產生提交檔案】 ---
print("\n--- 7. 正在用「最強的邏輯斯迴歸」產生提交檔案... ---")
test_predictions_lr_tuned = best_lr_model.predict(X_test)
submission_lr_tuned = pd.DataFrame({
    "PassengerId": test_passenger_ids,
    "Survived": test_predictions_lr_tuned
})
submission_lr_tuned.to_csv('submission_logistic_tuned.csv', index=False)

print("\n🎉 --- 最終提交檔案 submission_logistic_tuned.csv 建立完成！ --- 🎉")